# SAMap Cross-Species Analysis

Template notebook using `labcore.integration` and `labcore.scrnaseq`.
Edit the **Configuration** cell, then run all.

**Kernel:** `samap` (see `environment_samap.yml`)

## 1 · Configuration

In [1]:
import os

# ---------------------------------------------------------------------------
# Species
# ---------------------------------------------------------------------------
# Add or remove entries to change the number of species in the analysis.
# Keys are the short IDs SAMap uses internally (must match the BLAST map filenames).

species_files = {
    "ml": "/data/Mlig_scRNA_Seq/Francisco/Mlig_scRNASeq_atlas/results/objs/Ciliated_Neural_integrated_merged.h5ad",
    "sm": "/data/Mlig_scRNA_Seq/Francisco/Mlig_scRNASeq_atlas/data/ref_atlas/S_mediterranea/S_mediterranea_Neural_cells.h5ad",
    "sc": "/data/Mlig_scRNA_Seq/Francisco/Mlig_scRNASeq_atlas/data/ref_atlas/S_mansoni/S_mansoni_Neuron_cells.h5ad",
}

# Column in .obs that holds cluster / cell-type labels for each species
cluster_keys = {
    "ml": "neural_cluster",
    "sm": "Cluster.ID",
    "sc": "cell_type",
}

# Human-readable names for plot legends
species_display = {
    "ml": "M. lignano",
    "sm": "S. mediterranea",
    "sc": "S. mansoni",
}

# Colors for each species
species_colors = {
    "ml": "#377EB8",
    "sm": "#E41A1C",
    "sc": "#008000",
}

# Left-to-right order for Sankey diagrams
sankey_order = ["sm", "ml", "sc"]

# ---------------------------------------------------------------------------
# SAMap
# ---------------------------------------------------------------------------

samap_backend = "cpu"

samap_cell_type = "Ciliated_Neural"

maps_dir      = "/data/Mlig_scRNA_Seq/Francisco/Mlig_scRNASeq_atlas/data/ref_atlas/maps/"          # directory with BLAST / homology maps
run_samap     = True                       # False = load pre-computed object

# ---------------------------------------------------------------------------
# Gene-pair expression overlaps
# ---------------------------------------------------------------------------
# TSV with one row per gene pair. Required columns: one per species key
# (e.g. "ml", "sm", "sc") containing gene IDs. Leave a cell blank / NaN
# to exclude that species from a specific pair.
# Optional columns:
#   output_prefix  — filename prefix (defaults to gene IDs joined by "_")
#   label          — human-readable description (used only as a comment)
#
# Example row:
#   label    ml                sm                    sc          output_prefix
#   macpiwi  Mlig455-019997    dd-Smed-v4-15930-0-1              overlap_macpiwi
#
gene_pairs_file = "None"  # set None to skip this section

# ---------------------------------------------------------------------------
# Output directories
# ---------------------------------------------------------------------------
out_dir     = "/data/Mlig_scRNA_Seq/Francisco/Mlig_scRNASeq_atlas/results/cross_spp_annotation/results/"
figures_dir = os.path.join(out_dir, "figures/", samap_cell_type)
tables_dir  = os.path.join(out_dir, "tables/", samap_cell_type)
objs_dir    = os.path.join(out_dir, "objs/", samap_cell_type)

for d in [figures_dir, tables_dir, objs_dir]:
    os.makedirs(d, exist_ok=True)

# ---------------------------------------------------------------------------
# Analysis options
# ---------------------------------------------------------------------------
describe_data     = True   # print a structural summary of each h5ad
run_umap_sweep    = False   # sweep over UMAP min_dist values (exploratory)
align_thr         = 0.1    # mapping-score threshold (Sankey, CellTypeTriangles)
n_top             = 0      # n_top for get_mapping_scores (0 = keep all)
embedding_key     = "X_umap_samap"

## 2 · Imports

In [2]:
import warnings

import numpy as np
import pandas as pd
import scanpy as sc

# SAMap 3.x: all public symbols exported from the top-level package.
# samap.mapping and samap.analysis no longer exist as importable submodules.
from samap import (
    SAMAP,
    get_mapping_scores,
    GenePairFinder,
    CellTypeTriangles,
    load_samap,   # replaces pickle.load
    save_samap,   # replaces pickle.dump
)

from samap_extension import plotting as splt
from samap_extension import utils

## 3 · (Optional) Inspect input data

In [3]:
if describe_data:
    for sp, path in species_files.items():
        print(f"\n{'='*60}")
        print(f"Species: {species_display.get(sp, sp)}  ({sp})")
        print(f"{'='*60}")
        adata = sc.read_h5ad(path)
        utils.describe_adata(adata)
        del adata


Species: M. lignano  (ml)
Shape: 5,765 cells × 59,995 genes

Cell metadata (obs):
                                         orig.ident  nCount_RNA  nFeature_RNA  \
11_unsorted_3dph_TGGTTAGTCCGCTGTT  11_unsorted_3dph      3341.0          1492   
11_unsorted_3dph_GCACATATCGCCAGAC  11_unsorted_3dph      3282.0          1518   
11_unsorted_3dph_GGTAACTGTGCCTTCT  11_unsorted_3dph      3039.0          1368   
11_unsorted_3dph_ACGTAGTCAATCTCTT  11_unsorted_3dph      2989.0          1350   
11_unsorted_3dph_CTGCCATGTTCAGCTA  11_unsorted_3dph      2969.0          1541   

                                          sample_id   Stage      ProjectID  \
11_unsorted_3dph_TGGTTAGTCCGCTGTT  11_unsorted_3dph  3_days  Mlig_scRNASeq   
11_unsorted_3dph_GCACATATCGCCAGAC  11_unsorted_3dph  3_days  Mlig_scRNASeq   
11_unsorted_3dph_GGTAACTGTGCCTTCT  11_unsorted_3dph  3_days  Mlig_scRNASeq   
11_unsorted_3dph_ACGTAGTCAATCTCTT  11_unsorted_3dph  3_days  Mlig_scRNASeq   
11_unsorted_3dph_CTGCCATGTTCAGCTA  11_un

## 4 · Run or load SAMap

In [ ]:
samap_obj_path = os.path.join(objs_dir, "SAMap_cross_species_annotation.obj")

if run_samap:
    sm = SAMAP(species_files, f_maps=maps_dir, backend="cpu")
    sm.run()
    save_samap(sm, samap_obj_path)
    print(f"SAMap object saved to: {samap_obj_path}")
else:
    sm = load_samap(samap_obj)
    print(f"SAMap object loaded from: {samap_obj_path}")

INFO: Using backend: cpu
INFO: Processing data ml from: /data/Mlig_scRNA_Seq/Francisco/Mlig_scRNASeq_atlas/results/objs/Ciliated_Neural_integrated_merged.h5ad
INFO: Running SAM algorithm
INFO: Iteration: 0, Convergence: 1.000000
INFO: Iteration: 1, Convergence: 0.902862
INFO: Elapsed time: 5.05 seconds
INFO: Not updating the manifold...
INFO: Processing data sm from: /data/Mlig_scRNA_Seq/Francisco/Mlig_scRNASeq_atlas/data/ref_atlas/S_mediterranea/S_mediterranea_Neural_cells.h5ad
INFO: Running SAM algorithm
INFO: Iteration: 0, Convergence: 1.000000
INFO: Iteration: 1, Convergence: 0.824150
INFO: Iteration: 2, Convergence: 0.011037
INFO: Elapsed time: 17.93 seconds
INFO: Not updating the manifold...
INFO: Processing data sc from: /data/Mlig_scRNA_Seq/Francisco/Mlig_scRNASeq_atlas/data/ref_atlas/S_mansoni/S_mansoni_Neuron_cells.h5ad
INFO: Running SAM algorithm
INFO: Iteration: 0, Convergence: 1.000000
INFO: Iteration: 1, Convergence: 0.789986
INFO: Elapsed time: 11.39 seconds
INFO: Not up

## 5 · Mapping scores

In [ ]:
D, MappingTable = get_mapping_scores(sm, cluster_keys, n_top=n_top)

# Save mapping table
table_path = os.path.join(tables_dir, "mapping_scores.tsv")
pd.DataFrame(MappingTable).to_csv(table_path, sep="\t")
print(f"Mapping table saved to: {table_path}")

## 6 · Joint UMAP — coloured by species

In [ ]:
fig, ax = integration.square_scatter(
    sm,
    COLORS=species_colors,
    species_labels=species_display,
    figsize=(8, 8),
    dpi=600,
    s=60,
    alpha=0.5,
)
fig.savefig(
    os.path.join(figures_dir, "umap_by_species.png"),
    dpi=600,
    bbox_inches="tight",
)

## 7 · (Optional) UMAP `min_dist` sweep

Fits fresh UMAPs on the SAMap-aligned PCA space across a range of
`min_dist` values. Useful for choosing layout parameters before
committing to a final embedding. Enable with `run_umap_sweep = True`.

In [ ]:
import umap as umap_lib

if run_umap_sweep:
    min_dists = [0.01, 0.05, 0.1, 0.3, 0.5, 0.8]
    X_joint = np.vstack([
        sm.sams[sp].adata.obsm["X_pca"] for sp in sm.sams
    ])

    for md in min_dists:
        reducer = umap_lib.UMAP(
            n_components=2, min_dist=md, spread=1.0,
            metric="euclidean", random_state=0,
        )
        embedding = reducer.fit_transform(X_joint)

        start = 0
        for sp in sm.sams:
            n = sm.sams[sp].adata.n_obs
            key = f"X_umap_samap_mindist_{md}"
            sm.sams[sp].adata.obsm[key] = embedding[start:start + n]
            start += n

        fig, ax = integration.square_scatter(
            sm,
            COLORS=species_colors,
            species_labels=species_display,
            figsize=(6, 6),
            dpi=150,
            s=20,
            alpha=0.5,
        )
        ax.set_title(f"min_dist={md}")
        fig.savefig(
            os.path.join(figures_dir, f"umap_sweep_mindist_{md}.png"),
            dpi=150, bbox_inches="tight",
        )

## 8 · Joint UMAP — coloured by cell type per species

Produces one figure per species, highlighting that species' cluster
labels against all other cells in gray.

In [ ]:
for sp, col in cluster_keys.items():
    out_path = os.path.join(figures_dir, f"umap_labels_{sp}.png")
    fig, ax = integration.plot_joint_umap(
        sm,
        species=sp,
        label_col=col,
        embedding_key=embedding_key,
        figsize=(8, 8),
        dpi=300,
        save_to=out_path,
    )
    print(f"Saved: {out_path}")

## 9 · Sankey diagram

Uses `sankey_plot_3` for 3 species (two-panel layout keeping the
middle column aligned) and falls back to a single-panel Sankey for
2 species.

In [ ]:
n_species = len(species_files)
two_panel = n_species == 3

sankey = integration.sankey_plot_3(
    MappingTable,
    species_order=sankey_order,
    align_thr=align_thr,
    two_panel=two_panel,
    save_to=os.path.join(figures_dir, "sankey.html"),
    height=900,
    width_each=650,   # ignored when two_panel=False
    width=1200,       # used when two_panel=False
)
sankey

## 10 · Gene pair finder

In [ ]:
gpf = GenePairFinder(sm, keys=cluster_keys)
gene_pairs = gpf.find_all(align_thr=align_thr)

gpf_path = os.path.join(objs_dir, "gene_pair_finder.obj")
save_samap(gpf, gpf_path)

pairs_path = os.path.join(tables_dir, "gene_pairs.tsv")
gene_pairs.to_csv(pairs_path, sep="\t")
print(f"Gene pairs saved to: {pairs_path}")
gene_pairs.head(20)

## 11 · Expression overlap plots

Reads gene pairs from `gene_pairs_file` (TSV defined in Configuration).
Columns must include one entry per species key; blank / NaN cells exclude
that species from the pair. An `output_prefix` column is optional.

In [ ]:
if gene_pairs_file is not None:
    gp_df = pd.read_csv(gene_pairs_file, sep="\t", dtype=str).fillna("")

    for _, row in gp_df.iterrows():
        # Build genes dict, skipping species with no gene assigned
        genes = {
            sp: row[sp]
            for sp in species_files
            if sp in row.index and row[sp] != ""
        }
        if len(genes) < 2:
            print(f"Skipping row (fewer than 2 species with genes): {dict(row)}")
            continue

        # Build output prefix: use column if present, else join gene IDs
        if "output_prefix" in row.index and row["output_prefix"] != "":
            prefix = os.path.join(figures_dir, row["output_prefix"])
        else:
            prefix = os.path.join(figures_dir, "overlap_" + "_".join(genes.values()))

        # Restrict colors to species actually in this pair
        pair_colors = {sp: species_colors[sp] for sp in genes}

        integration.save_expression_overlap_plot(
            sm=sm,
            genes=genes,
            colors=pair_colors,
            species_labels=species_display,
            overlap_color="#FFEE8C",
            figsize=(5, 5),
            dpi=600,
            point_size=20,
            save_pdf=False,
            output_prefix=prefix,
            show=True,
        )
        print(f"Saved: {prefix}.png")
else:
    print("gene_pairs_file is None — skipping expression overlap plots.")

## 12 · Cell type triangles

In [ ]:
triangles = CellTypeTriangles(sm, cluster_keys, align_thr=align_thr)

tri_path = os.path.join(tables_dir, "cell_type_triangles.tsv")
triangles.to_csv(tri_path, sep="\t")
print(f"Cell type triangles saved to: {tri_path}")
triangles.head(20)